# Diabetes Screening Adherence — SCPR Eligible Population

Goal: from the SCPR "Alive" population extract, derive the count of **residents aged 40-100 who were alive within the 2021-01 to 2024-12 reference window** — the population eligible to be invited for SCPR diabetes screening. Target check value: **2,125,423**.

This notebook processes `ICAL229_SCPR_Alive_1.csv` (batch1) first. The filtering logic is wrapped in a function so it can be re-run for the remaining 7 files once this one is validated.

In [ ]:
import pandas as pd

s3_str = "s3://entrust-prd-moh-250040b/common-data/cr1/batch1/"

In [ ]:
scr1 = pd.read_csv(s3_str + "ICAL229_SCPR_Alive_1.csv", nrows=2)
scr1.head()

In [ ]:
scr1 = pd.read_csv(s3_str + "ICAL229_SCPR_Alive_1.csv")
scr1.shape

In [ ]:
scr1['UIN_identifiable'].nunique()

In [ ]:
## missing values per column
print("missing values per column:")
print(scr1.isnull().sum())

## 1. Check categorical codes before filtering

Confirm the actual `ResidentialStatus` codes in the data (e.g. is PR really coded `PR`, or something else like `P`/`S`) before hard-coding a filter list.

In [ ]:
print(scr1['ResidentialStatus'].value_counts(dropna=False))
print()
print(scr1['Gender'].value_counts(dropna=False))
print()
print(scr1['ReferencePeriod'].min(), "to", scr1['ReferencePeriod'].max())

## 2. Reusable eligibility filter

- `resident_codes`: update this list once the cell above confirms the actual codes for Citizen / PR.
- `DateofBirth` is `MM-YYYY` only (no day), so age is computed from year/month relative to each row's `ReferencePeriod`.
- "Alive between the 2 reference dates" = the row's `ReferencePeriod` falls within 2021-01 to 2024-12 (per your confirmation that the field is month-level).
- Result is deduplicated on `UIN_identifiable` since a person can appear in multiple monthly reference rows.

In [ ]:
resident_codes = ['C', 'PR']  # TODO: confirm against value_counts() output above
period_start = '2021-01-01'
period_end = '2024-12-31'
age_min, age_max = 40, 100


def compute_eligible_population(df, resident_codes=resident_codes,
                                 period_start=period_start, period_end=period_end,
                                 age_min=age_min, age_max=age_max):
    df = df.copy()

    df['DOB_parsed'] = pd.to_datetime(df['DateofBirth'], format='%m-%Y')
    df['RefPeriod_parsed'] = pd.to_datetime(df['ReferencePeriod'])

    df['Age'] = (df['RefPeriod_parsed'].dt.year - df['DOB_parsed'].dt.year) - \
        (df['RefPeriod_parsed'].dt.month < df['DOB_parsed'].dt.month).astype(int)

    mask_period = df['RefPeriod_parsed'].between(period_start, period_end)
    mask_age = df['Age'].between(age_min, age_max)
    mask_resident = df['ResidentialStatus'].isin(resident_codes)

    eligible = df.loc[mask_period & mask_age & mask_resident]
    eligible_unique = eligible.drop_duplicates(subset='UIN_identifiable')

    return eligible_unique


eligible_batch1 = compute_eligible_population(scr1)
print("Eligible unique residents (batch1):", eligible_batch1['UIN_identifiable'].nunique())

## 3. Sanity-check against the target (2,125,423)

If the count doesn't match, check in order: (a) `resident_codes` list, (b) whether `period_start`/`period_end` should instead require a person to appear in *both* the earliest and latest reference snapshot rather than any row within the window, (c) whether rows with `InvalidAddressTag == 'Y'` should be excluded.

In [ ]:
target = 2_125_423
actual = eligible_batch1['UIN_identifiable'].nunique()
print(f"target={target:,}  actual={actual:,}  diff={actual - target:,}")

## 4. Persist the eligible UIN list

Save the deduplicated eligible-population UINs so they can be joined against the actual screening/claims files (the remaining 7 CSVs) later to compute adherence = screened / eligible.

In [ ]:
eligible_batch1[['UIN_identifiable', 'Age', 'Gender', 'ResidentialStatus', 'RefPeriod_parsed']].to_parquet(
    "eligible_scpr_batch1.parquet", index=False
)

## 5. Once batch1 checks out — loop over the remaining files

Uncomment and adjust paths once the batch1 target count is confirmed. Combines all batches, then dedups on UIN across all of them for the final overall eligible-population count.

In [ ]:
# file_paths = [
#     "s3://entrust-prd-moh-250040b/common-data/cr1/batch1/ICAL229_SCPR_Alive_1.csv",
#     "s3://entrust-prd-moh-250040b/common-data/cr1/batch1/ICAL229_SCPR_Alive_2.csv",
#     # ... remaining files
# ]
#
# eligible_frames = []
# for path in file_paths:
#     df = pd.read_csv(path)
#     eligible_frames.append(compute_eligible_population(df))
#
# eligible_all = pd.concat(eligible_frames, ignore_index=True).drop_duplicates(subset='UIN_identifiable')
# print("Eligible unique residents (all batches):", eligible_all['UIN_identifiable'].nunique())